# Python File Reading & Opening Tutorial

This tutorial explains how to **open and read different kinds of files in Python**:
text files, JSON, JSON Lines, CSV, and binary files.
It is written to be read **top to bottom**.

---

## 1. The basic rule for opening files

Always use the `with open(...)` pattern.

```python
    with open("file.txt", "r") as f:
        data = f.read()
```
Why this is important:
- the file is closed automatically
- safer than manual `open()` / `close()`
- prevents file-handle leaks

---

## 2. Opening text files (`.txt`)

- Text files contain plain text.
- When reading text files, Python must convert **bytes on disk into characters in memory**.
- This conversion is controlled by the **encoding**.
- UTF-8 is the safe, modern default and supports characters from all languages.

---


### 2.1 Read the entire file at once

```python
with open("example.txt", "r", encoding="utf-8") as f:
    text = f.read()
```
What this does:
- Opens the file in read mode
- Decodes the file using UTF-8
- Reads the entire file into memory as a single string
- Stores it in the variable `text` (type: str)

When this is appropriate:
- Small text files
- Configuration files
- Short documents
- Files you are sure fit comfortably in memory

Why this can be dangerous:
- Large files (MB–GB) will:
  - consume a lot of memory
  - slow down your program
  - possibly crash your process

Rule of thumb:
Use `f.read()` only when you are confident the file is small.

---

### 2.2 Read line by line (recommended)

```python
with open("example.txt", "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()     # remove whitespace and newline
        if not line:            # skip empty lines
            continue
        print(line)
```

What this does:
- Opens the file safely with UTF-8 decoding
- Reads one line at a time
- Processes each line immediately
- Never loads the entire file into memory

Why this is better:
- Memory efficient
- Safe for very large files
- Scales well to real-world data

This is the preferred pattern for:
- Large text files
- PMC XML files
- Logs and corpora
- Production pipelines

---

### 2.3 Mental model (important)

- f.read() → load everything at once
- for line in f → stream the file safely

If you are unsure about file size, always choose line-by-line reading.


## 3. Opening JSON files (`.json`)

A `.json` file usually contains:
- one dictionary
- or one list

Example file content:

```python
    {
        "pmid": 12345,
        "title": "Example paper"
    }
```
---

### 3.1 Reading a JSON file

```python
    import json

    with open("example.json", "r", encoding="utf-8") as f:
        data = json.load(f)
```
Result:
- `data` is a `dict` or `list`
- entire file is loaded at once

---

## 4. JSON Lines / NDJSON (`.jsonl`)

JSON Lines format means:
**one JSON object per line**

Example file:
```python
    {"pmid": 1, "title": "A"}
    {"pmid": 2, "title": "B"}
```
---

### 4.1 Reading JSONL safely (best practice)
```python
    import json

    with open("example.jsonl", "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                record = json.loads(line)
                print(record)
            except json.JSONDecodeError:
                print("Skipping bad JSON line")
```
Why this pattern matters:
- one bad line does NOT crash the file
- safe for large datasets
- very common in data pipelines

---

## 5. Opening CSV files (`.csv`)

CSV files store tabular data.

---

### 5.1 Reading CSV rows as dictionaries

    import csv

    with open("example.csv", "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            print(row)

Each `row` is a dictionary:
- keys = column names
- values = cell values

---

## 6. Opening binary files (`.pdf`, `.zip`, `.bin`, images)

Binary files are **not text**.

---

### 6.1 Reading a binary file

    with open("example.pdf", "rb") as f:
        data = f.read()

Important:
- use `"rb"` mode
- do NOT specify encoding
- data is bytes, not strings

---

## 7. Common file modes (memorize)

    "r"   → read text
    "rb"  → read binary
    "w"   → write (overwrite)
    "a"   → append
    "r+"  → read + write

---

## 8. Handling errors when opening files

Files can fail to open due to:
- missing file
- permissions
- corrupted content

---

### 8.1 Safe pattern with try / except

    try:
        with open("example.txt", "r", encoding="utf-8") as f:
            text = f.read()
    except FileNotFoundError:
        print("File not found")
    except PermissionError:
        print("Permission denied")

---

## 9. Mental model (important)

- Text files → read line by line
- JSON → `json.load()` (whole file)
- JSONL → `json.loads()` per line
- CSV → `csv.DictReader`
- Binary → `"rb"` mode only

---

## 10. Golden rules

- Always use `with open(...)`
- Use UTF-8 for text unless you know otherwise
- Never load huge files all at once
- Catch parsing errors, not everything
